## Model Implementation
<hr>
Using our model to make song recommendations.

In [ ]:
pip install lyricsgenius; spotipy

In [ ]:
import pandas as pd
import numpy as np
import joblib
import pickle
import lyricsgenius
import requests
import time
import re
from collections import Counter
from tensorflow.keras.models import load_model
from scipy.sparse import load_npz, csr_matrix, vstack
from sklearn.metrics.pairwise import cosine_similarity

<b>Loading our TF-IDF model and vocabulary</b>

In [ ]:
model = load_model("lyric_tag_model.keras")
tfidf = joblib.load("tfidf_transformer.joblib")

with open("vocab_tags.pkl", "rb") as f:
    metadata = pickle.load(f)
    vocab = metadata["vocab"]
    selected_tags = metadata["selected_tags"]

catalog_rec = pd.read_pickle("catalog_rec.pkl")
X_rec = load_npz("X_rec.npz")

In [ ]:
catalog_probs = model.predict(X_rec, batch_size=512, verbose=1)
catalog_tag_masks = (catalog_probs >= 0.15)

275/275 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


### Song-by-song recommendations with subset (1000) of user track history

In [ ]:
def recommend_for_live_track(
    live_tfidf_vector, 
    model=model, 
    catalog=catalog_rec, 
    features=X_rec, 
    catalog_masks=catalog_tag_masks,
    top_n=3, 
    discovery_n=3
):
    scores = cosine_similarity(live_tfidf_vector, features).ravel()
    ranked_idx = np.argsort(scores)[::-1]
    
    top_idx = ranked_idx[:top_n]
    standard_recs = catalog.iloc[top_idx][["title", "artist_name"]].copy()
    standard_recs["lyric_similarity"] = scores[top_idx]
    standard_recs["rec_type"] = "Closest"
    
    live_probs = model.predict(live_tfidf_vector, verbose=0).ravel()
    live_tag_mask = (live_probs >= 0.15)
    
    if np.any(live_tag_mask):
        shared_tag_counts = catalog_masks.dot(live_tag_mask.astype(int))
        candidate_mask = shared_tag_counts > 0
    else:
        candidate_mask = np.ones(len(catalog), dtype=bool)
        
    candidate_mask[top_idx] = False
    valid_candidate_idx = [idx for idx in ranked_idx if candidate_mask[idx]]
    
    discovery_pool_idx = valid_candidate_idx[5:80]
    safe_discovery_n = min(discovery_n, len(discovery_pool_idx))
    
    if safe_discovery_n > 0:
        discovery_idx = np.random.choice(discovery_pool_idx, size=safe_discovery_n, replace=False)
        discovery_idx = discovery_idx[np.argsort(scores[discovery_idx])[::-1]]
        
        discovery_recs = catalog.iloc[discovery_idx][["title", "artist_name"]].copy()
        discovery_recs["lyric_similarity"] = scores[discovery_idx]
        discovery_recs["rec_type"] = "Discovery"
        
        return pd.concat([standard_recs, discovery_recs]).reset_index(drop=True)
        
    return standard_recs.reset_index(drop=True)

In [ ]:
url = "http://ws.audioscrobbler.com/2.0/"
username = "<REDACTED>"
api_key = "<REDACTED>"

In [ ]:
def get_recent_1000_lastfm_tracks(username, api_key):
    track_list = []
    page = 1
    
    print(f"Fetching recent Last.fm history for {username}...")
    
    while page <= 5:
        params = {
            "method": "user.getrecenttracks",
            "user": username,
            "api_key": api_key,
            "format": "json",
            "limit": 200, 
            "page": page
        }
        
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        print(f"Processing page {page} of 5...")
            
        for track in data.get('recenttracks', {}).get('track', []):
            if '@attr' in track and track['@attr'].get('nowplaying') == 'true':
                continue
                
            track_list.append({
                "artist": track['artist']['#text'],
                "title": track['name'],
                "album": track['album']['#text']
            })
            
        page += 1
        time.sleep(0.25)
        
    return pd.DataFrame(track_list)

df_history = get_recent_1000_lastfm_tracks(username, api_key)
print(f"\nSuccessfully loaded {len(df_history)} tracks into df_history!")

In [39]:
display(df_history)

,artist,title,album
0,Igorrr,Daemoni,AMEN
1,Allt,Aquila,Aquila
2,Ankor,MADARA · endless dream,MADARA · endless dream
3,Gojira,Mea Culpa (Ah! Ça ira!),Mea Culpa (Ah! Ça ira!)
4,gore.,Babylon,Babylon
...,...,...,...
995,Keep Close,Bury The Pain,Bury The Pain
996,Dead Lakes,hold back,daydreamer
997,Blessthefall,mallxcore,Gallows
998,I See Stars,SPLIT,THE WHEEL


#### Cleaning and transforming the lyrics pulled from the Genius API into 5000-feature TF-IDF vectors to align with the training vocabulary

In [ ]:
def lyrics_to_tfidf(raw_lyrics, vocab, tfidf_transformer):
    cleaned = re.sub(r'\[.*?\]', ' ', raw_lyrics)    
    cleaned = re.sub(r'[^a-z\'\s]', ' ', cleaned.lower())
    tokens = cleaned.split()
    
    if isinstance(vocab, (list, np.ndarray)):
        vocab_to_idx = {word: i for i, word in enumerate(vocab)}
    else:
        vocab_to_idx = vocab
        
    n_features = len(vocab_to_idx)
    counts = Counter(tokens)
    row_indices = []
    col_indices = []
    values = []
    
    for word, count in counts.items():
        if word in vocab_to_idx:
            row_indices.append(0)
            col_indices.append(vocab_to_idx[word])
            values.append(count)
            
    count_matrix = csr_matrix(
        (values, (row_indices, col_indices)),
        shape=(1, n_features),
        dtype=np.float32
    )
    
    tfidf_vector = tfidf_transformer.transform(count_matrix)
    
    return tfidf_vector

In [ ]:
genius = lyricsgenius.Genius("<REDACTED>")
genius.verbose = False
genius.timeout = 15
genius.retries = 3

for index, row in df_history.head(3).iterrows():
    print(f"\n--- Processing: {row['title']} by {row['artist']} ---")
    song = genius.search_song(row['title'], row['artist'])
    
    if song is None:
        print("Lyrics not found on Genius. Skipping...")
        continue
        
    raw_lyrics = song.lyrics
    live_tfidf_vector = lyrics_to_tfidf(raw_lyrics, vocab, tfidf)
    tag_probs = model.predict(live_tfidf_vector, verbose=0).ravel()
    top_tag_indices = np.argsort(tag_probs)[::-1]

    predicted_tags = [
        selected_tags[i] for i in top_tag_indices 
        if tag_probs[i] >= 0.15
    ][:5]
    
    print(f"Predicted Tags: {predicted_tags}")
    recommendations = recommend_for_live_track(live_tfidf_vector, top_n=3, discovery_n=3)
    print("Recommendations:")
    display(recommendations[['rec_type', 'title', 'artist_name', 'lyric_similarity']])


--- Processing: Daemoni by Igorrr ---
Predicted Tags: ['rock', 'pop']
Recommendations:


,rec_type,title,artist_name,lyric_similarity
0,Closest,Rap Inconscient,La Fouine,0.416093
1,Closest,Intro,Horkyze Slyze,0.374125
2,Closest,New Africa,Youssou N'Dour,0.350343
3,Discovery,Sunadokei,L'Arc~en~Ciel,0.220812
4,Discovery,Bamako,Youssou N'Dour,0.218153
5,Discovery,Tiptoe,Goldfrapp,0.210312



--- Processing: Aquila by Allt ---
Predicted Tags: ['metal', 'metalcore', 'hardcore', 'rock', 'death metal']
Recommendations:


,rec_type,title,artist_name,lyric_similarity
0,Closest,Bring It On Golden Pond,Pretty Girls Make Graves,0.298222
1,Closest,Darkness Undying,Nox,0.295353
2,Closest,When Darkness Falls (Album Version),Killswitch Engage,0.287016
3,Discovery,The Waiting Is Over,Frolic,0.229020
4,Discovery,Turned Inside Out (Reissue) (Album Version),Obituary,0.227845
5,Discovery,Surrounding,SECRET SPHERE,0.221972



--- Processing: MADARA · endless dream by Ankor ---
Predicted Tags: ['metal', 'heavy metal', 'rock', 'progressive metal', 'power metal']
Recommendations:


,rec_type,title,artist_name,lyric_similarity
0,Closest,Beneath The Encasing Of Ashes,As I Lay Dying,0.292587
1,Closest,Endless,Dave Gahan,0.291518
2,Closest,Endless,Technicolour,0.289416
3,Discovery,Clone,Gojira,0.230460
4,Discovery,The Rose,Bianca Ryan,0.227663
5,Discovery,The Martyr,BLEED THE SKY,0.227162


### User Profile recommendations with full user track history

In [ ]:
def get_full_lastfm_history(username, api_key, limit=200):
    track_list = []
    page = 1
    total_pages = 1
    print(f"Fetching full Last.fm history for the user...")
    
    while page <= total_pages:
        params = {
            "method": "user.getrecenttracks",
            "user": username,
            "api_key": api_key,
            "format": "json",
            "limit": limit,
            "page": page
        }
        
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if page == 1:
            total_pages = int(data['recenttracks']['@attr']['totalPages'])
            total_tracks = data['recenttracks']['@attr']['total']
            print(f"Found {total_tracks} total tracks across {total_pages} pages.")
            
        if page % 5 == 0 or page == total_pages:
            print(f"Processing page {page} of {total_pages}...")
            
        for track in data.get('recenttracks', {}).get('track', []):
            if '@attr' in track and track['@attr'].get('nowplaying') == 'true':
                continue
                
            track_list.append({
                "artist": track['artist']['#text'],
                "title": track['name'],
                "album": track['album']['#text']
            })
            
        page += 1
        time.sleep(0.2)
        
    return pd.DataFrame(track_list)

df_full_history = get_full_lastfm_history(username, api_key)
print(f"\nSuccessfully loaded {len(df_full_history)} tracks!")

In [40]:
display(df_full_history)

,artist,title,album
0,Igorrr,Daemoni,AMEN
1,Allt,Aquila,Aquila
2,Ankor,MADARA · endless dream,MADARA · endless dream
3,Gojira,Mea Culpa (Ah! Ça ira!),Mea Culpa (Ah! Ça ira!)
4,gore.,Babylon,Babylon
...,...,...,...
63100,Purity Ring,push pull,another eternity
63101,Melanie Martinez,Cry Baby,Cry Baby (Deluxe)
63102,Purity Ring,bodyache,Bodyache
63103,Broods,Everytime,Evergreen


<b>In order to handle a much larger Scrobble history, we:</b><br>
<ol>
<li>Fetch the user's top 50 artists, which should represent most of their listening history</li>
<li>Grab the top 3 tags for each artist</li>
<li>Find the 5 most dominant genres overall</li>
<li>Filter all artists based on those top genres</li>
<li>Filter all of the Scrobble history to ONLY include tracks by these artists</li>
<li>Remove duplicates and sample 150 random tracks from this pool</li>
</ol>

In [ ]:
params = {
    "method": "user.gettopartists", 
    "user": username, 
    "api_key": api_key, 
    "format": "json", 
    "limit": 50,
    "period": "overall"
}
response = requests.get(url, params=params).json()
top_artists = [artist['name'] for artist in response.get('topartists', {}).get('artist', [])]

genre_counts = Counter()
artist_genres = {}

print("Analyzing your top artists to determine core genres...")
for artist in top_artists:
    params = {"method": "artist.gettoptags", "artist": artist, "api_key": api_key, "format": "json"}
    tag_data = requests.get(url, params=params).json()
    
    tags = [tag['name'].lower() for tag in tag_data.get('toptags', {}).get('tag', [])[:3]]
    artist_genres[artist] = tags
    genre_counts.update(tags)
    time.sleep(0.1)

top_user_genres = [genre for genre, count in genre_counts.most_common(5)]
print(f"Your Top Genres: {top_user_genres}\n")

core_artists = [
    artist for artist, tags in artist_genres.items() 
    if any(g in top_user_genres for g in tags)
]

df_core_history = df_full_history[df_full_history['artist'].isin(core_artists)]
unique_core_tracks = df_core_history.drop_duplicates(subset=['title', 'artist']).reset_index(drop=True)
sampled_tracks = unique_core_tracks.sample(n=150, random_state=42).reset_index(drop=True)

print(f"Sampled 150 unique tracks strictly from your top genres!")
display(sampled_tracks.head())

Analyzing your top artists to determine core genres...
Your True Top Genres: ['japanese', 'j-rock', 'visual kei', 'metalcore', 'rock']

Sampled 150 unique tracks strictly from your core genres!


,artist,title,album
0,DIR EN GREY,鱗,
1,Evanescence,A New Way to Bleed,Evanescence (Deluxe Version)
2,RENTRER EN SOI,AMONGST FOOLISH ENEMIES,AIN SOPH AUR
3,DIMLIM,Mad [K],CHEDOARA
4,BUCK-TICK,変身(REBORN),狂った太陽


In [ ]:
tfidf_vectors = []
print(f"\nProcessing lyrics for {len(sampled_tracks)} tracks...")

for idx, row in sampled_tracks.iterrows():
    song = genius.search_song(row['title'], row['artist'])
    
    if song and song.lyrics:
        vec = lyrics_to_tfidf(song.lyrics, vocab, tfidf)
        tfidf_vectors.append(vec)

if tfidf_vectors:
    user_history_matrix = vstack(tfidf_vectors)
    
    all_track_probs = model.predict(user_history_matrix, verbose=0)
    user_tag_probs = all_track_probs.mean(axis=0)
    
    user_profile_vector = np.asarray(user_history_matrix.max(axis=0).todense())
    
    print(f"User taste profile successfully built!")

#### For the user profile recommendations, we:
<ol>
<li>Ensure the vector is correctly shaped for scikit-learn (1, 5000)</li>
<li>Calculate cosine similarity between the user's profile and the entire catalog</li>
<li>Get similar candidates from the closest matches to the user's overall profile</li>
<li>Use the pre-calculated averaged tag probabilities from Late Fusion</li>
<li>Filter discovery candidates by requiring at least one shared predicted tag</li>
<li>Sample discovery tracks from a slightly lower tier</li>
</ol>

In [ ]:
def recommend_for_user_profile(
    user_profile_vector, 
    user_tag_probs,
    model=model, 
    catalog=catalog_rec, 
    features=X_rec, 
    catalog_masks=catalog_tag_masks,
    top_n=5, 
    discovery_n=5
):
    if user_profile_vector.ndim == 1:
        user_profile_vector = user_profile_vector.reshape(1, -1)
        
    scores = cosine_similarity(user_profile_vector, features).ravel()
    ranked_idx = np.argsort(scores)[::-1]    
    top_idx = ranked_idx[:top_n]
    standard_recs = catalog.iloc[top_idx][["title", "artist_name"]].copy()
    standard_recs["lyric_similarity"] = scores[top_idx]
    standard_recs["rec_type"] = "Similar"
    user_tag_mask = (user_tag_probs >= 0.15)
    
    if np.any(user_tag_mask):
        shared_tag_counts = catalog_masks.dot(user_tag_mask.astype(int))
        candidate_mask = shared_tag_counts > 0
    else:
        candidate_mask = np.ones(len(catalog), dtype=bool)
        
    candidate_mask[top_idx] = False
    valid_candidate_idx = [idx for idx in ranked_idx if candidate_mask[idx]]
    
    discovery_pool_idx = valid_candidate_idx[10:100]
    safe_discovery_n = min(discovery_n, len(discovery_pool_idx))
    
    if safe_discovery_n > 0:
        discovery_idx = np.random.choice(discovery_pool_idx, size=safe_discovery_n, replace=False)
        discovery_idx = discovery_idx[np.argsort(scores[discovery_idx])[::-1]]
        
        discovery_recs = catalog.iloc[discovery_idx][["title", "artist_name"]].copy()
        discovery_recs["lyric_similarity"] = scores[discovery_idx]
        discovery_recs["rec_type"] = "Discovery"
        
        return pd.concat([standard_recs, discovery_recs]).reset_index(drop=True)
        
    return standard_recs.reset_index(drop=True)

In [60]:
user_recs = recommend_for_user_profile(user_profile_vector, user_tag_probs, top_n=5, discovery_n=5)
display(user_recs[['rec_type', 'title', 'artist_name', 'lyric_similarity']])

Top Predicted User Taste Tags: ['rock', 'alternative', 'indie', 'alternative rock', 'pop', 'metal']



,rec_type,title,artist_name,lyric_similarity
0,Similar,Safety Dance,Men Without Hats,0.379708
1,Similar,Night Of The Living Dead,The Misfits,0.368399
2,Similar,My Life,Deviates,0.366968
3,Similar,The Kids Aren't Alright,The Offspring,0.329671
4,Similar,You Never Know,Immortal Technique,0.327765
5,Discovery,What's On Your Mind,Eric B. & Rakim,0.294672
6,Discovery,Street Hassle,Lou Reed,0.281763
7,Discovery,Where Are You,Manafest,0.280330
8,Discovery,Paradise By The Dashboard Light (Live) (2006 D...,Meat Loaf,0.277691
9,Discovery,Opera,Floetry,0.276224


Since the original Last.fm dataset was compiled in 2011 and does not contain recent tracks, we suppliment that missing data by using Spotify's API to search for songs similar to the songs recommended by our model.<br><br>

As the Spotify data is <b>not</b> being used to train the AI model, its usage is allowed.<br><br>

Spotify, however, no longer allows access to genre data across many of its endpoints. Therefore, we return to using the Last.fm API to include tags/genre information in the final output.

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id="<REDACTED>",
    client_secret="<REDACTED>"
))

In [ ]:
def get_unplayed_spotify_track(artist_name, df_history, sp_client):
    played_tracks = set(
        df_history[df_history['artist'].str.lower() == artist_name.lower()]['title'].str.lower()
    )
    
    params = {"method": "artist.gettoptags", "artist": artist_name, "api_key": api_key, "format": "json"}
    
    try:
        tag_data = requests.get(url, params=params, timeout=5).json()
        tags = [tag['name'].title() for tag in tag_data.get('toptags', {}).get('tag', [])[:3]]
        artist_genres = ", ".join(tags) if tags else "None listed"
    except Exception:
        artist_genres = "None listed"
        
    query = f'artist:"{artist_name}"'
    results = sp_client.search(q=query, type='track', limit=10)
    tracks = results.get('tracks', {}).get('items', [])
    
    for track in tracks:
        if track['name'].lower() not in played_tracks:
            return {
                "title": track['name'],
                "artist": track['artists'][0]['name'],
                "spotify_url": track['external_urls']['spotify'],
                "artist_genres": artist_genres
            }
            
    if tracks:
        return {
            "title": tracks[0]['name'],
            "artist": tracks[0]['artists'][0]['name'],
            "spotify_url": tracks[0]['external_urls']['spotify'],
            "artist_genres": artist_genres
        }
        
    return None

In [87]:
def modernize_local_recs(local_recs_df, df_history, sp_client):
    fresh_recs = []
    
    for idx, row in local_recs_df.iterrows():
        artist = row['artist_name']
        track_info = get_unplayed_spotify_track(artist, df_history, sp_client)
        
        if track_info:
            fresh_recs.append({
                "rec_type": row['rec_type'],
                "title": track_info['title'],
                "artist_name": track_info['artist'],
                "genres": track_info['artist_genres'], 
                "lyric_similarity": row['lyric_similarity'],
                "spotify_url": track_info['spotify_url']
            })
            
    return pd.DataFrame(fresh_recs)

In [88]:
user_recs = recommend_for_user_profile(user_profile_vector, user_tag_probs, top_n=5, discovery_n=5)
fresh_user_recs = modernize_local_recs(user_recs, df_full_history, sp)

display(fresh_user_recs[['rec_type', 'title', 'artist_name', 'genres', 'lyric_similarity']])

Top Predicted User Taste Tags: ['rock', 'alternative', 'indie', 'alternative rock', 'pop', 'metal']



,rec_type,title,artist_name,genres,lyric_similarity
0,Similar,The Safety Dance,Men Without Hats,"New Wave, 80S, Synthpop",0.379708
1,Similar,Halloween,Misfits,"Horror Punk, Punk, Punk Rock",0.368399
2,Similar,Come With Me,Deviates,"Punk Rock, Punk, Skate Punk",0.366968
3,Similar,The Kids Aren't Alright,The Offspring,"Punk Rock, Punk, Rock",0.329671
4,Similar,Dance With The Devil,Immortal Technique,"Hip-Hop, Rap, Underground Hip-Hop",0.327765
5,Discovery,Guatauba,Plan B,"Hip-Hop, British, Grime",0.305590
6,Discovery,WHINE N STICK,Kid Billionz,"Progressive Rock, Neo-Prog, Neo Progressive",0.301070
7,Discovery,Babies,Pulp,"Britpop, Indie, British",0.283993
8,Discovery,Hall of the Mountain King,Savatage,"Heavy Metal, Progressive Metal, Power Metal",0.283500
9,Discovery,My Wine In Silence,My Dying Bride,"Doom Metal, Gothic Metal, Death Metal",0.277951
